# Fine-tune the speaker model on Samrómur Children

Phase 3, Task 4 of `docs/superpowers/plans/2026-10-28-phase3-fine-tuning.md` in
[Talat5563/presence-app](https://github.com/Talat5563/presence-app). Starts from WeSpeaker's
`voxceleb_resnet34_LM`, the model the app ships. Fine-tunes it on Samrómur Children with
`training/conf/samromur_resnet34_ft.yaml`, picks the epoch on 152 held-out speakers, and exports
a drop-in `speaker_embedding.onnx`.

**Licences.**
- Samrómur Children is CC BY 4.0, on one condition: do not attempt to identify any speaker. Nothing here does.
- The WeSpeaker checkpoint is CC BY 4.0.
- CHILDES and MyST are non-commercial and never come near this notebook.

**How to run**
1. Create a notebook on Kaggle and import this file (File → Import Notebook).
2. Under **Settings**, set Accelerator to **GPU T4 x2** (a P100 also works) and turn **Internet**
   on. Internet needs a phone-verified Kaggle account.
3. Choose **Save Version → Save & Run All (Commit)**. The run is headless and Kaggle allows up to
   12 hours. Training stops itself at `TRAIN_BUDGET_HOURS`, so scoring and export still fit.
4. If training stopped early, add the previous version's output as an input (Add Input → Your
   Work), set `RESUME_FROM` to its `out/exp` folder, and commit again.
5. Download the version's output folder `out/`:
   - `speaker_embedding_ft.onnx`
   - `scores/holdout/epoch_scores.csv`
   - `run.json`
   - `exp/train.log`
   - `exp/config.yaml`
   - `pip-freeze.txt`

Everything is pinned:
- this repository's commit (recorded in `run.json`);
- WeSpeaker's commit;
- the checkpoint and the dataset revisions, checked by SHA-256;
- Python 3.11 and every package, via `training/requirements-kaggle.txt`.

In [ ]:
# Parameters
PRESENCE_REPO = "https://github.com/Talat5563/presence-app.git"
PRESENCE_REF = "childes-stand-in-corpus"  # branch or commit; the resolved commit goes in run.json
WESPEAKER_COMMIT = "9fecd6cb4f47475d01761d87c826298dff4ef18c"
CHECKPOINT_REVISION = "f0c48c298fd835726c27956a5d617bad7115627e"
CHECKPOINT_SHA256 = {
    "avg_model": "9872b375f2c6a3851ca471cbbf59e06efd23a627d78bf5872e1f0269fd298449",
    "config.yaml": "3cf7d3243464cd939083e29d2be65c2abcdd954c1a64559bad73b74ffdb0db3e",
}
NUM_EPOCHS = None          # None keeps the config's 15
TRAIN_BUDGET_HOURS = 9.0   # stop training here so scoring and export fit in Kaggle's 12 h
RESUME_FROM = None         # e.g. "/kaggle/input/<previous-version>/out/exp"
BASELINE_TEST_EER_HARD = 0.1126  # the app's own number on Samrómur test (training/README.md)

In [ ]:
import csv, hashlib, json, os, platform, shutil, signal, subprocess, sys, time, urllib.request
from pathlib import Path

SRC, VENV, DATA = Path("/tmp/src"), Path("/tmp/venv"), Path("/tmp/data")
OUT = Path("/kaggle/working/out")
for d in (SRC, DATA, OUT):
    d.mkdir(parents=True, exist_ok=True)
PRESENCE, WESPEAKER = SRC / "presence", SRC / "wespeaker"
T = PRESENCE / "training"
PY, TORCHRUN = VENV / "bin" / "python", VENV / "bin" / "torchrun"
SAM, CKPT = DATA / "samromur", DATA / "voxceleb_resnet34_LM"
ENV = dict(os.environ, PYTHONPATH=str(WESPEAKER))


def sh(cmd, **kwargs):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, env=ENV, **kwargs)


def run_python(code):
    """Runs code in the training venv (a file, so quotes in it never meet the shell)."""
    script = DATA / "check.py"
    script.write_text(code)
    sh(f"{PY} {script}")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            digest.update(block)
    return digest.hexdigest()


record_path = OUT / "run.json"
record = json.loads(record_path.read_text()) if record_path.exists() else {}


def save_record(**fields):
    record.update(fields)
    record_path.write_text(json.dumps(record, indent=2))


gpus = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                      capture_output=True, text=True).stdout.split("\n")
gpus = [g.strip() for g in gpus if g.strip()]
assert gpus, "No GPU visible: Settings -> Accelerator -> GPU"
free_gb = shutil.disk_usage(DATA).free / 1e9
assert free_gb > 25, f"only {free_gb:.0f} GB free under {DATA}: the data needs about 14 GB and the venv about 6"
save_record(gpus=gpus, cpus=os.cpu_count(), kaggle_python=platform.python_version(),
            started=time.strftime("%Y-%m-%d %H:%M:%S"))
print(gpus, f"{os.cpu_count()} CPUs, {free_gb:.0f} GB free")

## Code and environment

In [ ]:
if not PRESENCE.exists():
    sh(f"git clone -q {PRESENCE_REPO} {PRESENCE} && git -C {PRESENCE} checkout -q {PRESENCE_REF}")
if not WESPEAKER.exists():
    sh(f"git clone -q https://github.com/wenet-e2e/wespeaker.git {WESPEAKER} "
       f"&& git -C {WESPEAKER} checkout -q {WESPEAKER_COMMIT}")
save_record(presence_commit=subprocess.check_output(["git", "-C", str(PRESENCE), "rev-parse", "HEAD"], text=True).strip(),
            wespeaker_commit=subprocess.check_output(["git", "-C", str(WESPEAKER), "rev-parse", "HEAD"], text=True).strip())
print(record["presence_commit"], record["wespeaker_commit"])

In [ ]:
# libsox backs torchaudio's sox_effects, which WeSpeaker's speed perturbation calls.
sh("apt-get -qq update > /dev/null && apt-get -qq install -y sox libsox-dev > /dev/null")
UV = f"{sys.executable} -m uv"
sh(f"{sys.executable} -m pip install -q uv==0.9.5")
if not PY.exists():
    sh(f"{UV} venv -q {VENV} --python 3.11")
    sh(f"{UV} pip install -q --python {VENV} --index-strategy unsafe-best-match "
       f"--build-constraint {T}/build-constraints-kaggle.txt -r {T}/requirements-kaggle.txt")
sh(f"{UV} pip freeze --python {VENV} > {OUT}/pip-freeze.txt")

check = f"""
import sys, torch, torchaudio
assert torch.cuda.is_available(), "CUDA is not visible to the venv's torch"
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
x = torch.randn(256, 256, device="cuda")
assert torch.isfinite(x @ x).all()
print("torch", torch.__version__, "CUDA", torch.version.cuda, names)
wav = torch.randn(1, 16000)
out, _ = torchaudio.sox_effects.apply_effects_tensor(wav, 16000, [["speed", "1.1"], ["rate", "16000"]])
assert abs(out.shape[1] - 16000 / 1.1) < 200, out.shape
print("sox speed perturbation works")
sys.path.insert(0, "{WESPEAKER}")
import wespeaker.bin.train
print("WeSpeaker's training script imports")
"""
run_python(check)

## Starting checkpoint and data

In [ ]:
CKPT.mkdir(parents=True, exist_ok=True)
base = f"https://huggingface.co/Wespeaker/wespeaker-voxceleb-resnet34-LM/resolve/{CHECKPOINT_REVISION}"
for name, digest in CHECKPOINT_SHA256.items():
    if not (CKPT / name).exists():
        urllib.request.urlretrieve(f"{base}/{name}", CKPT / name)
    assert sha256(CKPT / name) == digest, f"{name} does not match its pinned SHA-256"
# Named for its role, so the scores table cannot confuse it with WeSpeaker's averaged models.
BASELINE = CKPT / "baseline.pt"
if not BASELINE.exists():
    shutil.copy(CKPT / "avg_model", BASELINE)
print("checkpoint verified")

In [ ]:
# About 15 minutes: 6.6 GB download, hash checks, unpacking, and a format check of every file.
SAM.mkdir(parents=True, exist_ok=True)
if not (SAM / ".prepared").exists():
    sh(f"{PY} {T}/prepare_samromur.py {SAM} --fetch --fetch-test")
    sh(f"{PY} {T}/make_trials.py {SAM} holdout")
    sh(f"{PY} {T}/make_trials.py {SAM} test")
    sh(f"{PY} {WESPEAKER}/tools/make_raw_list.py {SAM}/wespeaker/train/wav.scp "
       f"{SAM}/wespeaker/train/utt2spk {SAM}/wespeaker/train/raw.list")
    for archive in SAM.glob("*.tar.gz"):
        archive.unlink()  # the unpacked FLACs are all training needs
    (SAM / ".prepared").touch()
sh(f"wc -l {SAM}/wespeaker/train/raw.list {SAM}/wespeaker/holdout/wav.scp {SAM}/test.scp")

## Checks before training

WeSpeaker loads `model_init` non-strictly: a key mismatch only logs a warning, and it would then
train from random weights. The first check loads the checkpoint the way `train.py` does and
confirms every backbone tensor arrived.

The second check scores the unchanged checkpoint on Samrómur test. It must reproduce the
app's own Kotlin result, 11.26% hard EER, or the holdout numbers that pick the epoch cannot be
trusted. This is the published baseline, not a selection. Fine-tuned models are never scored on
test here.

In [ ]:
CONFIG = T / "conf" / "samromur_resnet34_ft.yaml"
check = f"""
import sys, torch, yaml
sys.path.insert(0, "{WESPEAKER}")
from wespeaker.models.speaker_model import get_speaker_model
from wespeaker.utils.checkpoint import load_checkpoint
config = yaml.safe_load(open("{CONFIG}"))
model = get_speaker_model(config["model"])(**config["model_args"])
load_checkpoint(model, "{BASELINE}")
state = torch.load("{BASELINE}", map_location="cpu", weights_only=True)
loaded = model.state_dict()
backbone = {{k for k in state if not k.startswith("projection.")}}
assert set(loaded) == backbone, sorted(set(loaded) ^ backbone)[:5]
assert all(torch.equal(loaded[k], state[k]) for k in loaded)
print(f"model_init loads all {{len(loaded)}} backbone tensors; the head starts fresh")
"""
run_python(check)

In [ ]:
test_scores = OUT / "scores" / "test" / "baseline_test.csv"
if not test_scores.exists():
    sh(f"{PY} {T}/score_epochs.py --wespeaker {WESPEAKER} --config {CONFIG} --list {SAM}/test.scp "
       f"--trials {SAM}/trials_test.txt --out {test_scores} {BASELINE}")
row = next(csv.DictReader(open(test_scores)))
save_record(baseline_test_eer_hard=float(row["eer_hard"]), baseline_test_eer_random=float(row["eer_random"]))
assert abs(float(row["eer_hard"]) - BASELINE_TEST_EER_HARD) < 0.002, (
    f"the scorer gives {float(row['eer_hard']):.2%} where the app gives {BASELINE_TEST_EER_HARD:.2%}")
print(f"scorer reproduces the app's baseline: EER hard {float(row['eer_hard']):.2%}")

## Train

In [ ]:
import yaml

config = yaml.safe_load(open(CONFIG))
if NUM_EPOCHS:
    config["num_epochs"] = NUM_EPOCHS
config["dataloader_args"]["num_workers"] = max(1, (os.cpu_count() or 4) // len(gpus))
RUN_CONFIG = OUT / "train_config.yaml"
RUN_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
EXP = OUT / "exp"
if RESUME_FROM and not EXP.exists():
    shutil.copytree(RESUME_FROM, EXP, symlinks=True)


def epochs_done():
    models = EXP / "models"
    return sorted(int(p.stem.split("_")[1]) for p in models.glob("model_*.pt")) if models.exists() else []


last = max(epochs_done(), default=0)
if last == 0 and EXP.exists():
    shutil.rmtree(EXP)  # a run that died before finishing its first epoch
if last < config["num_epochs"]:
    gpu_list = "[" + ",".join(str(i) for i in range(len(gpus))) + "]"
    cmd = (f"cd {WESPEAKER} && {TORCHRUN} --standalone --nnodes=1 --nproc_per_node={len(gpus)} "
           f"wespeaker/bin/train.py --config {RUN_CONFIG} --exp_dir {EXP} --gpus '{gpu_list}' "
           f"--num_avg {config['num_avg']} --data_type raw "
           f"--train_data {SAM}/wespeaker/train/raw.list --train_label {SAM}/wespeaker/train/utt2spk "
           f"--model_init {BASELINE}"
           + (f" --checkpoint {EXP}/models/model_{last}.pt" if last else ""))
    print("$", cmd, flush=True)
    started = time.time()
    process = subprocess.Popen(cmd, shell=True, env=ENV, start_new_session=True)
    try:
        process.wait(timeout=TRAIN_BUDGET_HOURS * 3600)
        assert process.returncode == 0, f"training failed with exit code {process.returncode}"
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait()
        print(f"stopped at the {TRAIN_BUDGET_HOURS} h budget; commit again with RESUME_FROM to continue")
    save_record(train_hours=round(record.get("train_hours", 0) + (time.time() - started) / 3600, 2))
done = epochs_done()
save_record(epochs_done=len(done), num_epochs=config["num_epochs"])
print(f"{len(done)} of {config['num_epochs']} epochs done")

## Pick the checkpoint on the holdout

Scores the unchanged model, every epoch and, once training is complete, the average of the last
`num_avg` epochs. The pick is the lowest hard EER: the same gender, at most a year apart in age.
That is the closest Samrómur comes to telling siblings apart.

In [ ]:
holdout_scores = OUT / "scores" / "holdout" / "epoch_scores.csv"
complete = len(done) == config["num_epochs"]
avg = EXP / "models" / "avg_model.pt"
if complete and not avg.exists():
    sh(f"{PY} {WESPEAKER}/wespeaker/bin/average_model.py --dst_model {avg} "
       f"--src_path {EXP}/models --num {config['num_avg']}")
candidates = [BASELINE] + [EXP / "models" / f"model_{e}.pt" for e in done] + ([avg] if complete else [])
scored = {r["checkpoint"] for r in csv.DictReader(open(holdout_scores))} if holdout_scores.exists() else set()
todo = [str(c) for c in candidates if c.name not in scored]
if todo:
    sh(f"{PY} {T}/score_epochs.py --wespeaker {WESPEAKER} --config {RUN_CONFIG} "
       f"--list {SAM}/wespeaker/holdout/wav.scp --trials {SAM}/trials_holdout.txt --out {holdout_scores} " + " ".join(todo))
rows = list(csv.DictReader(open(holdout_scores)))
for r in rows:
    print(f"{r['checkpoint']:16} EER hard {float(r['eer_hard']):7.2%}   random {float(r['eer_random']):7.2%}   "
          f"minDCF hard {float(r['min_dcf_hard']):.3f}")
best = min((r for r in rows if r["checkpoint"] != "baseline.pt"), key=lambda r: float(r["eer_hard"]))
baseline = next(r for r in rows if r["checkpoint"] == "baseline.pt")
save_record(chosen=best["checkpoint"], chosen_holdout=best, baseline_holdout=baseline, complete=complete)
print(f"chosen: {best['checkpoint']}, holdout EER hard {float(best['eer_hard']):.2%} "
      f"against {float(baseline['eer_hard']):.2%} for the unchanged model")

## Export and verify the ONNX file

Exports with WeSpeaker's `export_onnx.py`, the exporter behind the shipped model. Then checks the
tensor contract the app relies on: input `feats [B, T, 80]` and output `embs [B, 256]`. Finally it
embeds the holdout through `embed_onnx.py`, the app's front end in Python. Every file must match
the PyTorch embedding at cosine 0.999 or better, and the holdout EER must match too.

In [ ]:
ONNX = OUT / "speaker_embedding_ft.onnx"
sh(f"{PY} {WESPEAKER}/wespeaker/bin/export_onnx.py --config {EXP}/config.yaml "
   f"--checkpoint {EXP}/models/{best['checkpoint']} --output_model {ONNX}")
check = f"""
import onnx
m = onnx.load("{ONNX}")
dims = lambda v: [d.dim_param or d.dim_value for d in v.type.tensor_type.shape.dim]
inputs = [(i.name, dims(i)) for i in m.graph.input]
outputs = [(o.name, dims(o)) for o in m.graph.output]
assert inputs == [("feats", ["B", "T", 80])], inputs
assert outputs == [("embs", ["B", 256])], outputs
print("tensor contract holds:", inputs, outputs)
"""
run_python(check)
onnx_embeddings = OUT / "scores" / "holdout" / "onnx-embeddings.csv"
sh(f"{PY} {T}/embed_onnx.py {ONNX} {SAM}/wespeaker/holdout/wav.scp {onnx_embeddings}")
torch_embeddings = OUT / "scores" / "holdout" / f"{Path(best['checkpoint']).stem}-embeddings.csv"
sh(f"{PY} {T}/compare_embeddings.py {torch_embeddings} {onnx_embeddings}")
sh(f"{PY} {T}/score_trials.py {onnx_embeddings} {SAM}/trials_holdout.txt")
save_record(onnx_sha256=sha256(ONNX), finished=time.strftime("%Y-%m-%d %H:%M:%S"))
print(json.dumps(record, indent=2))

## Next, on the local machine (Tasks 5 and 6)

Back in the repository, with `out/` downloaded:

1. **Put the model in place.** Copy `speaker_embedding_ft.onnx` in place of
   `models/speaker_embedding.onnx` and run `./gradlew :shared:integrationTest`.
2. **Front-end parity.** Embed the Samrómur test list with `:harness:embedFiles` (Kotlin) and with
   `training/embed_onnx.py` (Python), then run `training/compare_embeddings.py` on the two.
3. **Samrómur test, once.** Score the Kotlin embeddings with `training/score_trials.py`.
4. **CHILDES.** Re-run the harness on the same manifests as the Task 0 baselines.